In [1]:
# 1. Desinstalar la versión estándar por si acaso
!pip uninstall -y ultralytics

# 2. Instalar la versión ESPECÍFICA de YOLOv12
!pip install git+https://github.com/sunsmarterjie/yolov12.git
!pip install fpdf

# 3. Instalamos FastAPI y herramientas de servidor (REEMPLAZA A STREAMLIT)
!pip install fastapi uvicorn python-multipart

  Cloning https://github.com/sunsmarterjie/yolov12.git to /tmp/pip-req-build-jflq0owu
  Running command git clone --filter=blob:none --quiet https://github.com/sunsmarterjie/yolov12.git /tmp/pip-req-build-jflq0owu
  Resolved https://github.com/sunsmarterjie/yolov12.git to commit 01a22c0603e0eaa6d9bd62120a391e744d92cea2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.3.63-py3-none-any.whl size=910519 sha256=2b820e9b4f6ac4cae97b5593462be444182d5dbfba786312bb6cd57e560e154d
  Stored in directory: /tmp/pip-ephem-wheel-cache-s7lqo1lt/wheels/51/14/51/9f3f73766e89100fb390b031d2290899b2fef7b57d9a74c6dc
Successfully built ultralytics
  Preparing metadata (setup.py) ... done
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=d6475f163d596d4188dd1a9ae7e651c853062553cf9447ec7ba09acfff1b9d06
  Stored in directory: /root/.cach

In [2]:
import shutil
import os
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 1. Copiar modelos desde Drive
modelos = ['best.pt', 'best_v8_sard.pt', 'best_yolov12.pt', 'best-RTDETR.pt']
for m in modelos:
    src = f'/content/drive/MyDrive/TFG/{m}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/{m}')
        print(f" Modelo {m} cargado en el entorno.")

# 2. Recuperar el historial de estadísticas correcto
csv_name = 'registro_analisis_v2.csv'
path_drive = f'/content/drive/MyDrive/TFG/{csv_name}'

if os.path.exists(path_drive):
    shutil.copy(path_drive, f'/content/{csv_name}')
    print(f" Archivo '{csv_name}' recuperado de Drive.")
else:
    print(f" No se encontró '{csv_name}' en Drive, se creará uno nuevo al analizar.")

Mounted at /content/drive
 Modelo best.pt cargado en el entorno.
 Modelo best_v8_sard.pt cargado en el entorno.
 Modelo best_yolov12.pt cargado en el entorno.
 Modelo best-RTDETR.pt cargado en el entorno.
 Archivo 'registro_analisis_v2.csv' recuperado de Drive.


In [3]:
%%writefile api.py
from fastapi import FastAPI, UploadFile, File, Form, BackgroundTasks
from fastapi.responses import FileResponse, Response, JSONResponse, StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from starlette.concurrency import run_in_threadpool
import cv2
import os
import time
import subprocess
import tempfile
import torch
import psutil
from datetime import datetime
import uuid
import numpy as np
from ultralytics import YOLO, RTDETR
from fpdf import FPDF
import json
import asyncio
import math
import threading
import csv

app = FastAPI(title="SAR AI Backend Pro - MultiTenant")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

DEVICE = 0 if torch.cuda.is_available() else "cpu"
LOG_FILE = 'registro_analisis_v2.csv'

# --- GESTIÓN DE SESIONES MULTI-USUARIO ---
sessions = {}
csv_lock = threading.Lock()

def update_session(session_id, progreso=None, fase=None):
    if session_id not in sessions:
        sessions[session_id] = {"progreso": 0, "fase": "INICIALIZANDO"}
    if progreso is not None: sessions[session_id]["progreso"] = progreso
    if fase is not None: sessions[session_id]["fase"] = fase

def get_session(session_id):
    return sessions.get(session_id, {"progreso": 0, "fase": "ESPERANDO CONEXION"})

# --- TRACKER ---
class PythonTacticalTracker:
    def __init__(self, w, h, max_paciencia=45):
        self.tracks = []
        self.next_id = 1
        self.max_paciencia = max_paciencia
        self.max_dist = max(w, h) * 0.50

    def calcular_distancia(self, boxA, boxB):
        cA = ((boxA[0]+boxA[2])/2, (boxA[1]+boxA[3])/2)
        cB = ((boxB[0]+boxB[2])/2, (boxB[1]+boxB[3])/2)
        return math.sqrt((cA[0]-cB[0])**2 + (cA[1]-cB[1])**2)

    def update(self, detecciones):
        for t in self.tracks:
            t['box'][0] += t['vx']; t['box'][1] += t['vy']
            t['box'][2] += t['vx']; t['box'][3] += t['vy']
            t['vx'] *= 0.2; t['vy'] *= 0.2
            t['age'] += 1

        matches = []
        for d_idx, det in enumerate(detecciones):
            for t_idx, t in enumerate(self.tracks):
                dist = self.calcular_distancia(det, t['box'])
                if dist < self.max_dist:
                    matches.append({'d': d_idx, 't': t_idx, 'score': 1 / (dist + 1)})

        matches = sorted(matches, key=lambda x: x['score'], reverse=True)
        matched_d, matched_t = set(), set()

        for m in matches:
            if m['d'] not in matched_d and m['t'] not in matched_t:
                matched_d.add(m['d']); matched_t.add(m['t'])
                track = self.tracks[m['t']]
                det = detecciones[m['d']]

                frames = max(1, track['age'])
                cxOld = (track['last_box'][0] + track['last_box'][2]) / 2
                cyOld = (track['last_box'][1] + track['last_box'][3]) / 2
                cxNew = (det[0] + det[2]) / 2
                cyNew = (det[1] + det[3]) / 2

                track['vx'] = ((cxNew - cxOld) / frames) * 0.4
                track['vy'] = ((cyNew - cyOld) / frames) * 0.4

                track['box'] = list(det[:4])
                track['last_box'] = list(det[:4])
                track['conf'] = det[4]
                track['age'] = 0

        for d_idx, det in enumerate(detecciones):
            if d_idx not in matched_d:
                self.tracks.append({
                    'id': self.next_id, 'box': list(det[:4]), 'last_box': list(det[:4]),
                    'conf': det[4], 'vx': 0, 'vy': 0, 'age': 0
                })
                self.next_id += 1

        self.tracks = [t for t in self.tracks if t['age'] <= self.max_paciencia]
        return self.tracks

# --- LOG DE TELEMETRÍA SEGURO ---
LOG_FILE_LOCAL = 'registro_analisis_v2.csv'
LOG_FILE_DRIVE = '/content/drive/MyDrive/TFG/registro_analisis_v2.csv'

def save_log_entry(tipo, duracion, fps, modelo, cpu):
    nueva_fila = {
        'ID': str(uuid.uuid4())[:8], 'Fecha': datetime.now().strftime("%Y-%m-%d"),
        'Hora': datetime.now().strftime("%H:%M:%S"), 'Tipo': tipo, 'Modelo': modelo,
        'Duracion_Sec': round(float(duracion), 2), 'FPS_Promedio': round(float(fps), 1),
        'CPU_Usage': round(float(cpu), 1)
    }
    with csv_lock:
        for target in [LOG_FILE_DRIVE, LOG_FILE_LOCAL]:
            try:
                exists = os.path.exists(target)
                with open(target, mode='a', newline='', encoding='utf-8') as f:
                    writer = csv.DictWriter(f, fieldnames=nueva_fila.keys())
                    if not exists:
                        writer.writeheader()
                    writer.writerow(nueva_fila)
                break  # Si Drive funciona, no hace falta escribir en local
            except Exception as e:
                print(f"[LOG] Fallo escribiendo en {target}: {e}")

def load_model_safe(path, default="best_yolov12.pt"):
    try:
        if os.path.exists(path):
            if "rtdetr" in path.lower(): return RTDETR(path)
            return YOLO(path)
        return YOLO(default)
    except Exception as e: return None

models_cache = {}
def get_model(name):
    config = { "YOLO v5": "best.pt", "YOLO v8": "best_v8_sard.pt", "YOLO v12": "best_yolov12.pt", "RT-DETR": "best-RTDETR.pt" }
    path = config.get(name, "best_yolov12.pt")
    if path not in models_cache: models_cache[path] = load_model_safe(path)
    return models_cache[path]

def generar_reporte_pdf(nombre_archivo, modelo, conf, duracion, total_objetivos, detecciones, imagenes, max_confianza, session_id):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", 'B', 16)
    pdf.set_text_color(255, 87, 34)
    pdf.cell(200, 10, txt="SISTEMA SAR - REPORTE DE INTELIGENCIA TACTICA", ln=True, align='C')
    pdf.set_font("Arial", size=10)
    pdf.set_text_color(50, 50, 50)
    pdf.ln(5)
    now = datetime.utcnow()
    pdf.cell(200, 6, txt=f"Fecha y Hora (UTC): {now.strftime('%Y-%m-%d %H:%M:%S')}", ln=True)
    pdf.cell(200, 6, txt=f"Archivo Origen: {nombre_archivo}", ln=True)
    pdf.cell(200, 6, txt=f"Arquitectura IA: {modelo}", ln=True)
    pdf.cell(200, 6, txt=f"ID de Mision (Sesion): {session_id}", ln=True)
    pdf.cell(200, 6, txt=f"Umbral de Busqueda: {conf}", ln=True)
    pdf.cell(200, 6, txt=f"Tiempo de Procesamiento: {duracion:.2f} segundos", ln=True)
    pdf.ln(5)
    pdf.set_font("Arial", 'B', 12)
    pdf.set_text_color(0, 0, 0)
    pdf.cell(200, 8, txt="RESUMEN DE CONTACTOS:", ln=True)
    pdf.set_font("Arial", size=10)

    pdf.cell(200, 6, txt=f"Objetivos Unicos Detectados: {total_objetivos}", ln=True)
    pdf.cell(200, 6, txt=f"Capturas en Evidencia Visual: {len(imagenes)} (Las de mayor confianza)", ln=True)
    pdf.cell(200, 6, txt=f"Confianza Maxima Alcanzada: {max_confianza:.1f}%", ln=True)

    if detecciones:
        pdf.ln(3)
        for d in detecciones:
            pdf.cell(200, 5, txt=f"  [>] Min {d['tiempo']} - Contacto (Confianza: {d['confianza']}%)", ln=True)

    if imagenes:
        pdf.add_page()
        pdf.set_font("Arial", 'B', 12)
        pdf.cell(200, 10, txt="EVIDENCIA VISUAL (TOP 6 DE LA MISION)", ln=True)
        pdf.ln(5)

        y_pos = 25
        for img_path in imagenes:
            if os.path.exists(img_path):
                img_cv = cv2.imread(img_path)
                h_img, w_img, _ = img_cv.shape

                if h_img > w_img:
                    img_h_pdf = 110
                    img_w_pdf = 110 * (w_img / h_img)
                    x_offset = (210 - img_w_pdf) / 2

                    if y_pos + img_h_pdf > 280:
                        pdf.add_page()
                        y_pos = 20

                    pdf.image(img_path, x=x_offset, y=y_pos, h=img_h_pdf)
                    y_pos += img_h_pdf + 10
                else:
                    img_w_pdf = 180
                    img_h_pdf = 180 * (h_img / w_img)
                    x_offset = 15

                    if y_pos + img_h_pdf > 280:
                        pdf.add_page()
                        y_pos = 20

                    pdf.image(img_path, x=x_offset, y=y_pos, w=img_w_pdf)
                    y_pos += img_h_pdf + 10

                try: os.remove(img_path)
                except: pass

    pdf.output(f"reporte_tactico_{session_id}.pdf")

# =========================================================================
# PROCESAMIENTO DE VÍDEO
# =========================================================================
def _process_video_optimized(input_path, original_filename, modelo_name, conf, session_id):
    update_session(session_id, 5, f"INFERENCIA TENSORIAL ({modelo_name})")
    raw_path = tempfile.NamedTemporaryFile(delete=False, suffix='.mp4').name
    ia_model = get_model(modelo_name)

    cap = cv2.VideoCapture(input_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_in = cap.get(cv2.CAP_PROP_FPS) or 30

    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    max_dim = 854

    if orig_w > orig_h:
        w = max_dim
        h = int(max_dim * (orig_h / orig_w))
    else:
        h = max_dim
        w = int(max_dim * (orig_w / orig_h))

    w = w - (w % 2)
    h = h - (h % 2)

    out_fps = fps_in / 2 if fps_in > 0 else 15
    out = cv2.VideoWriter(raw_path, cv2.VideoWriter_fourcc(*'mp4v'), out_fps, (w, h))

    tracker = PythonTacticalTracker(w, h, max_paciencia=45)

    start_t = time.time()
    count = 0
    cpu_track = []
    registro_detecciones = []
    registro_capturas = []

    max_confianza_global = 0.0
    ultimo_seg_guardado = -1
    total_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        if count % 2 == 0:
            frame_res = cv2.resize(frame, (w, h))
            res = ia_model.predict(frame_res, conf=conf, verbose=False, device=DEVICE)

            detecciones_frame = []
            if len(res[0].boxes) > 0:
                for box in res[0].boxes:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    c = float(box.conf[0])
                    if c > max_confianza_global: max_confianza_global = c
                    detecciones_frame.append([x1, y1, x2, y2, c])

            tracks = tracker.update(detecciones_frame)

            frame_has_valid_target = False
            best_score_in_frame = 0

            for t in tracks:
                if t['age'] == 0:
                    x1, y1, x2, y2 = map(int, t['box'])
                    score = t['conf'] * 100
                    if score > best_score_in_frame: best_score_in_frame = score
                    frame_has_valid_target = True

                    texto = f"[ID:{t['id']}] PER {score:.0f}%"
                    (tw, th), _ = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)

                    # 1. Dibujar Caja Exterior (Naranja SAR)
                    cv2.rectangle(frame_res, (x1, y1), (x2, y2), (34, 87, 255), 2, cv2.LINE_AA)

                    # 2. Dibujar fondo de texto (Negro Puro para contraste absoluto)
                    y_bg_top = max(0, y1 - th - 10)
                    cv2.rectangle(frame_res, (x1, y_bg_top), (x1 + tw + 10, y1), (0, 0, 0), -1)

                    # 3. Dibujar texto (Blanco Puro sobre fondo negro)
                    cv2.putText(frame_res, texto, (x1 + 5, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)

            total_count = tracker.next_id - 1
            if total_count > 0:
                texto_hud = f"OBJETIVOS UNICOS: {total_count}"
                # Panel negro puro para máxima legibilidad
                cv2.rectangle(frame_res, (15, 15), (320, 60), (0, 0, 0), -1)
                # Borde naranja
                cv2.rectangle(frame_res, (15, 15), (320, 60), (34, 87, 255), 2, cv2.LINE_AA)
                cv2.putText(frame_res, texto_hud, (30, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)

            out.write(frame_res)

            if frame_has_valid_target:
                seg_actual = int(count / fps_in)
                if seg_actual > ultimo_seg_guardado:
                    registro_detecciones.append({
                        "tiempo": f"{seg_actual//60:02d}:{seg_actual%60:02d}",
                        "clase": "PERSONA",
                        "confianza": round(best_score_in_frame, 1)
                    })

                    img_name = f"snap_{session_id}_{uuid.uuid4().hex[:4]}.jpg"
                    cv2.imwrite(img_name, frame_res)

                    registro_capturas.append({
                        "file": img_name,
                        "score": best_score_in_frame
                    })

                    registro_capturas = sorted(registro_capturas, key=lambda x: x['score'], reverse=True)
                    if len(registro_capturas) > 6:
                        peor_captura = registro_capturas.pop()
                        if os.path.exists(peor_captura["file"]):
                            os.remove(peor_captura["file"])

                    ultimo_seg_guardado = seg_actual

            if count % 10 == 0:
                cpu_track.append(psutil.cpu_percent())
                update_session(session_id, min(80, int((count / total_frames) * 80)))

        count += 1

    cap.release()
    out.release()

    dur = time.time() - start_t
    avg_cpu = sum(cpu_track)/len(cpu_track) if cpu_track else psutil.cpu_percent()
    save_log_entry("VIDEO", dur, (count/2)/dur if dur>0 else 0, modelo_name, avg_cpu)

    update_session(session_id, 85, "CODIFICANDO FORMATO WEB")
    final_path = raw_path.replace(".mp4", "_f.mp4")
    subprocess.run(["ffmpeg", "-y", "-i", raw_path, "-vcodec", "libx264", "-preset", "ultrafast", "-crf", "35", "-movflags", "+faststart", final_path], capture_output=True)
    if os.path.exists(raw_path): os.remove(raw_path)

    update_session(session_id, 95, "ENSAMBLANDO INFORME PDF")

    imagenes_top = [c["file"] for c in registro_capturas]
    generar_reporte_pdf(original_filename, modelo_name, conf, dur, total_count, registro_detecciones, imagenes_top, max_confianza_global * 100, session_id)

    update_session(session_id, 100, "SISTEMA LISTO")
    return final_path

# =========================================================================
# PROCESAMIENTO DE IMAGEN
# =========================================================================
def _process_imagen_optimized(input_path, original_filename, modelo_name, conf, session_id):
    update_session(session_id, 5, f"INFERENCIA TENSORIAL ({modelo_name})")
    raw_path = tempfile.NamedTemporaryFile(delete=False, suffix='.jpg').name
    ia_model = get_model(modelo_name)

    frame = cv2.imread(input_path)
    h_img, w_img, _ = frame.shape

    # --- ESCALADO DINÁMICO ---
    escala = max(1.0, min(w_img, h_img) / 800.0)
    grosor = max(2, int(2 * escala))
    fuente_tam = 0.5 * escala
    pad = int(5 * escala)

    start_t = time.time()
    res = ia_model.predict(frame, conf=conf, verbose=False, device=DEVICE)

    detecciones_frame = []
    max_confianza_global = 0.0

    if len(res[0].boxes) > 0:
        for box in res[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            c = float(box.conf[0])
            if c > max_confianza_global: max_confianza_global = c
            detecciones_frame.append([int(x1), int(y1), int(x2), int(y2), c])

    update_session(session_id, 50, "DIBUJANDO HUD TACTICO")
    for det in detecciones_frame:
        x1, y1, x2, y2, c = det
        score = c * 100
        texto = f"PER {score:.0f}%"
        (tw, th), _ = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, fuente_tam, grosor)

        # 1. Caja Exterior Naranja
        cv2.rectangle(frame, (x1, y1), (x2, y2), (34, 87, 255), grosor, cv2.LINE_AA)

        # 2. Fondo de texto Negro puro
        y_bg_top = max(0, y1 - th - (pad * 2))
        cv2.rectangle(frame, (x1, y_bg_top), (x1 + tw + (pad * 2), y1), (0, 0, 0), -1)

        # 3. Texto en Blanco
        cv2.putText(frame, texto, (x1 + pad, y1 - pad), cv2.FONT_HERSHEY_SIMPLEX, fuente_tam, (255, 255, 255), max(1, int(grosor/2)), cv2.LINE_AA)

    total_count = len(detecciones_frame)
    if total_count > 0:
        hud_pad = int(15 * escala)
        hud_w = int(320 * escala)
        hud_h = int(60 * escala)
        fuente_hud = 0.7 * escala

        # Fondo del contador en Negro puro
        cv2.rectangle(frame, (hud_pad, hud_pad), (hud_pad + hud_w, hud_pad + hud_h), (0, 0, 0), -1)
        cv2.rectangle(frame, (hud_pad, hud_pad), (hud_pad + hud_w, hud_pad + hud_h), (34, 87, 255), grosor, cv2.LINE_AA)
        cv2.putText(frame, f"OBJETIVOS UNICOS: {total_count}", (hud_pad + int(15*escala), hud_pad + int(38*escala)), cv2.FONT_HERSHEY_SIMPLEX, fuente_hud, (255, 255, 255), grosor, cv2.LINE_AA)

    cv2.imwrite(raw_path, frame)
    dur = time.time() - start_t
    cpu_usage = psutil.cpu_percent()
    save_log_entry("IMAGEN", dur, 0, modelo_name, cpu_usage)

    update_session(session_id, 90, "ENSAMBLANDO INFORME PDF")

    registro_detecciones = [{"tiempo": "00:00", "clase": "PERSONA", "confianza": round(det[4]*100, 1)} for det in detecciones_frame]

    img_pdf_path = f"snap_pdf_{session_id}_{uuid.uuid4().hex[:4]}.jpg"
    cv2.imwrite(img_pdf_path, frame)

    generar_reporte_pdf(original_filename, modelo_name, conf, dur, total_count, registro_detecciones, [img_pdf_path], max_confianza_global * 100, session_id)

    update_session(session_id, 100, "SISTEMA LISTO")
    return raw_path

# =========================================================================
# ENDPOINTS FastAPI
# =========================================================================

@app.post("/analizar_video/")
async def analizar_video(background_tasks: BackgroundTasks, file: UploadFile = File(...), modelo: str = Form("YOLO v12"), conf: float = Form(0.35), session_id: str = Form(...)):
    update_session(session_id, 0, "PREPARANDO NODO GPU")
    temp_in = tempfile.NamedTemporaryFile(delete=False, suffix='.mp4')
    temp_in.write(await file.read())
    temp_in.close()

    try:
        output = await run_in_threadpool(_process_video_optimized, temp_in.name, file.filename, modelo, conf, session_id)
        background_tasks.add_task(os.remove, temp_in.name)
        background_tasks.add_task(os.remove, output)
        return FileResponse(output, media_type="video/mp4")
    except Exception as e:
        update_session(session_id, 0, f"ERROR: {str(e)}")
        return JSONResponse(status_code=500, content={"error": str(e)})

@app.post("/analizar_imagen/")
async def analizar_imagen(background_tasks: BackgroundTasks, file: UploadFile = File(...), modelo: str = Form("YOLO v12"), conf: float = Form(0.35), session_id: str = Form(...)):
    update_session(session_id, 0, "PREPARANDO NODO GPU")
    temp_in = tempfile.NamedTemporaryFile(delete=False, suffix='.jpg')
    temp_in.write(await file.read())
    temp_in.close()

    try:
        output = await run_in_threadpool(_process_imagen_optimized, temp_in.name, file.filename, modelo, conf, session_id)
        background_tasks.add_task(os.remove, temp_in.name)
        background_tasks.add_task(os.remove, output)
        return FileResponse(output, media_type="image/jpeg")
    except Exception as e:
        update_session(session_id, 0, f"ERROR: {str(e)}")
        return JSONResponse(status_code=500, content={"error": str(e)})

@app.get("/stream_progreso/")
async def stream_progreso(session_id: str):
    async def event_generator():
        last_progreso, last_fase = -1, ""
        while True:
            estado = get_session(session_id)
            if estado["progreso"] != last_progreso or estado["fase"] != last_fase:
                last_progreso, last_fase = estado["progreso"], estado["fase"]
                yield json.dumps(estado) + "\n"
            if estado["progreso"] >= 100 or "ERROR" in estado["fase"]: break
            await asyncio.sleep(0.3)
    return StreamingResponse(event_generator(), media_type="application/x-ndjson")

@app.get("/descargar_informe/")
def descargar_informe(session_id: str):
    pdf_path = f"reporte_tactico_{session_id}.pdf"
    if os.path.exists(pdf_path):
        return FileResponse(pdf_path, media_type="application/pdf", filename="SAR_Reporte_Tactico.pdf")
    return JSONResponse(status_code=404, content={"error": "Informe no encontrado"})

@app.get("/estadisticas/")
def stats():
    for path in [LOG_FILE_DRIVE, LOG_FILE_LOCAL]:
        if os.path.exists(path):
            return FileResponse(path, media_type="text/csv", headers={
                "Cache-Control": "no-cache, no-store, must-revalidate",
                "Pragma": "no-cache", "Expires": "0"
            })
    return JSONResponse(status_code=404, content={"error": "Aun no hay datos"})

@app.get("/")
def root(): return {"status": "ONLINE", "gpu": torch.cuda.is_available()}

Writing api.py


In [4]:
!pip install pyngrok -q
from pyngrok import ngrok
import os
import time

# Limpiar procesos
os.system("pkill -9 -f uvicorn")

# token de Ngrok
ngrok.set_auth_token("3AOqn3S77UKkQp9XTb7JU3Gj9v9_3KA7AjQMH3JqX8qW6xqj6")

# Iniciar API
print(" Arrancando FastAPI...")
os.system("nohup uvicorn api:app --host 0.0.0.0 --port 8000 > fastapi.log 2>&1 &")
time.sleep(3)

# Conectar al dominio estático
public_url = ngrok.connect(8000, domain="groundable-unratable-macy.ngrok-free.dev").public_url

print(f"\n API EN LÍNEA Y CONECTADA AL FRONTEND: {public_url}")

 Arrancando FastAPI...

 API EN LÍNEA Y CONECTADA AL FRONTEND: https://groundable-unratable-macy.ngrok-free.dev
